This 2D model will:

take MRI slices as input
classify them into:
normal
abnormal
save the best model
evaluate accuracy, precision, recall, F1-score, confusion matrix

In [8]:
import os
import copy
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [10]:
#add the dataset path,
BASE_PATH = "/Users/apple/Desktop/BRAINIAC/archive"

In [11]:
train_dir = os.path.join(BASE_PATH, "train")
val_dir = os.path.join(BASE_PATH, "val")
test_dir = os.path.join(BASE_PATH, "test")

print("Train path:", train_dir)
print("Val path:", val_dir)
print("Test path:", test_dir)

Train path: /Users/apple/Desktop/BRAINIAC/archive/train
Val path: /Users/apple/Desktop/BRAINIAC/archive/val
Test path: /Users/apple/Desktop/BRAINIAC/archive/test


In [14]:
import os

for root, dirs, files in os.walk("/Users/apple/Desktop/BRAINIAC"):
    print(root)

/Users/apple/Desktop/BRAINIAC
/Users/apple/Desktop/BRAINIAC/BraTS2020_ValidationData
/Users/apple/Desktop/BRAINIAC/BraTS2020_ValidationData/MICCAI_BraTS2020_ValidationData
/Users/apple/Desktop/BRAINIAC/BraTS2020_ValidationData/MICCAI_BraTS2020_ValidationData/BraTS20_Validation_069
/Users/apple/Desktop/BRAINIAC/BraTS2020_ValidationData/MICCAI_BraTS2020_ValidationData/BraTS20_Validation_056
/Users/apple/Desktop/BRAINIAC/BraTS2020_ValidationData/MICCAI_BraTS2020_ValidationData/BraTS20_Validation_051
/Users/apple/Desktop/BRAINIAC/BraTS2020_ValidationData/MICCAI_BraTS2020_ValidationData/BraTS20_Validation_058
/Users/apple/Desktop/BRAINIAC/BraTS2020_ValidationData/MICCAI_BraTS2020_ValidationData/BraTS20_Validation_093
/Users/apple/Desktop/BRAINIAC/BraTS2020_ValidationData/MICCAI_BraTS2020_ValidationData/BraTS20_Validation_067
/Users/apple/Desktop/BRAINIAC/BraTS2020_ValidationData/MICCAI_BraTS2020_ValidationData/BraTS20_Validation_060
/Users/apple/Desktop/BRAINIAC/BraTS2020_ValidationData/MIC

In [12]:
IMG_SIZE = 224

train_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

val_test_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

In [13]:
train_dataset = datasets.ImageFolder(train_dir, transform=train_transforms)
val_dataset = datasets.ImageFolder(val_dir, transform=val_test_transforms)
test_dataset = datasets.ImageFolder(test_dir, transform=val_test_transforms)

print("Classes:", train_dataset.classes)
print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

FileNotFoundError: [Errno 2] No such file or directory: '/Users/apple/Desktop/BRAINIAC/archive/train'

In [ ]:
BATCH_SIZE = 16

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
images, labels = next(iter(train_loader))

plt.figure(figsize=(12, 6))
for i in range(min(6, len(images))):
    img = images[i].permute(1, 2, 0).numpy()
    img = (img * 0.5) + 0.5  # unnormalize
    plt.subplot(2, 3, i+1)
    plt.imshow(img)
    plt.title(train_dataset.classes[labels[i]])
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

print(model)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    running_correct = 0
    total = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        running_correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = running_correct / total
    return epoch_loss, epoch_acc

In [ ]:
def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    running_correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            running_correct += (preds == labels).sum().item()
            total += labels.size(0)

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = running_correct / total
    return epoch_loss, epoch_acc

In [ ]:
EPOCHS = 10
best_val_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())

train_losses, val_losses = [], []
train_accs, val_accs = [], []

MODEL_SAVE_PATH = "/content/drive/MyDrive/Brainiac/models/normal_2d_model.pt"
# If local:
# MODEL_SAVE_PATH = "Brainiac/models/normal_2d_model.pt"

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")
    print("-" * 50)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save(best_model_wts, MODEL_SAVE_PATH)
        print("Best model saved.\n")

print("Training complete.")
print("Best validation accuracy:", best_val_acc)

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.title("Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_accs, label="Train Accuracy")
plt.plot(val_accs, label="Val Accuracy")
plt.title("Accuracy Curve")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
model.eval()

In [ ]:
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc = accuracy_score(all_labels, all_preds)

print("Test Accuracy:", test_acc)
print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds, target_names=test_dataset.classes))
print("Confusion Matrix:\n")
print(confusion_matrix(all_labels, all_preds))

In [ ]:
def predict_single_image(model, image_path, class_names, device):
    transform = transforms.Compose([
        transforms.Grayscale(num_output_channels=3),
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ])

    from PIL import Image
    image = Image.open(image_path).convert("L")
    x = transform(image).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        output = model(x)
        probs = torch.softmax(output, dim=1)
        pred = torch.argmax(probs, dim=1).item()

    print("Predicted class:", class_names[pred])
    print("Probabilities:", probs.cpu().numpy())

MRI Scan (WORKFLOW FOR REFERENCE)
   
2D Screening Model

  Normal → Non-tumorous output
  
  Abnormal →
  send to 3D Brainiac pipeline
                         ↓


                 Swin UNETR Segmentation
                         ↓

               Attention + Entropy + CSS
                         ↓

                      Risk Model